# 05 检索文档Document [百炼]

Firework AI没有/v1/files接口

百炼有/v1/files接口

阿里云百炼平台不支持/vector_stores接口（隶属于 Assistants API v2），当可以手动创建知识库提供知识库id即可知识检索。

In [15]:
import os
from rich import print
from dotenv import load_dotenv
from semantic_kernel.agents import OpenAIResponsesAgent

In [16]:
load_dotenv("../.env/bailian", override=True)
model_id = os.getenv("OPENAI_RESPONSES_MODEL_ID")
model_id

'qwen3.5-plus'

In [17]:
client = OpenAIResponsesAgent.create_client()

### 代码上传 ❌

In [ ]:
pdf_file_path = "../python-samples/getting_started_with_agents/resources/employees.pdf"

with open(pdf_file_path, "rb") as file:
    file = await client.files.create(file=file, purpose="file-extract")
    print(f"[blue]{file.id}[/blue]")


FileObject(id='file-fe-5a6b451a2d024d80bda411e0', bytes=43422, created_at=1773731595, filename='employees.pdf', 
object='file', purpose='file-extract', status='processed', expires_at=None, status_details=None) ▶︎ 
file-fe-5a6b451a2d024d80bda411e0

In [21]:
await client.files.delete("file-fe-5a6b451a2d024d80bda411e0")

FileDeleted(id='file-fe-5a6b451a2d024d80bda411e0', deleted=True, object='file')

参考 [如何在阿里云百炼模型中使用vector_stores](https://help.aliyun.com/zh/model-studio/file-search)，使用前先创建知识库（百炼后台-应用-知识库）中手动创建 [知识库test](https://bailian.console.aliyun.com/cn-beijing?spm=a2c4g.11186623.0.0.64f3297fbrXhm1&tab=app#/knowledge-base)。

In [ ]:
vector_store = await client.vector_stores.create(
    name="responses_agent_file_search",
    file_ids=[file.id],
)

### 手动添加知识库ID ✅

cz2ujdvuld 是我创建的知识库test的ID，

In [27]:
file_search_tool = OpenAIResponsesAgent.configure_file_search_tool("cz2ujdvuld")

agent_doc = OpenAIResponsesAgent(
    ai_model_id=model_id,
    client=client,
    instructions="Find answers to the user's questions in the provided file.",
    name="FileSearch",
    # store_enabled=False,
    tools=[file_search_tool],
)

### 检索知识库

In [28]:
queries = [
    "By birthday, who is the youngest employee?",
    "Who works in sales?",
    "I have a customer request, who can help me?",
]

In [ ]:
thread = None
for user_input in queries:
    print(f"[red]User ▶︎ {user_input}[/red]")

    async for response in agent_doc.invoke(messages=user_input, thread=thread):
        print(f"[blue]Agent ▶︎ {response.content}[/blue]")

        thread = response.thread


# User: 'By birthday, who is the youngest employee?'

# Agent: Based on the employee list provided in the document, the youngest employee is **Teodor Britton**, whose 
birthday is **Jan. 9, 1997**.

# User: 'Who works in sales?'

# Agent: Based on the employee list, the following people work in sales:

*   **Mariam Jaslyn** (Sales representative)
*   **Hicran Bea** (Sales manager)
*   **Angelino Embla** (Sales representative)

# User: 'I have a customer request, who can help me?'

# Agent: Based on the employee list provided, there is no specific "Customer Service" or "Support" role listed. 
However, depending on the nature of the customer request, the following employees would be the most appropriate to 
help:

*   **For sales, product, or order inquiries:**
    *   **Mariam Jaslyn** (Sales representative)
    *   **Angelino Embla** (Sales representative)
    *   **Hicran Bea** (Sales manager)

*   **For billing or payment issues:**
    *   **Clemence Hadiya** (Accountant)
    *   **Teodor Britton** (Accountant)

*   **For general inquiries or routing:**
    *   **Brendon Hilpert** (Administrative assistant)

If the request is general, **Brendon Hilpert** (Administrative assistant) or **Hicran Bea** (Sales manager) would 
likely be the best starting points.